In [1]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

tf.random.set_seed(42)

# Text constants (same as 03_glove_rnn_text.ipynb)
STOPWORDS = set(stopwords.words('english'))
MAX_LEN   = 50
VOCAB_SIZE = 10000
EMBED_DIM  = 100

# Image constants (same as 03_cnn_mobilenet.ipynb)
IMG_SIZE = (128, 128)
IMG_ROOT = '../data/Images'

# Known best hyperparams from unimodal runs — skip re-tuning to keep runtime manageable
LSTM_UNITS  = 32   # best from 03_glove_rnn_text.ipynb
DENSE_UNITS = 256  # best from 03_cnn_mobilenet.ipynb

## Build paired DataFrame

Join LabeledText.csv with image paths on the numeric file ID (`1.txt` ↔ `1.jpg`).
This gives one row per tweet with both its caption and image path.

In [2]:
# Build image lookup: file_id → image path
folder_to_label = {'Negative': 'negative', 'Neutral': 'neutral', 'positive': 'positive'}
img_records = []
for folder in folder_to_label:
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            fid = int(os.path.splitext(fname)[0])
            img_records.append({'file_id': fid, 'img_path': os.path.join(folder_path, fname)})
img_df = pd.DataFrame(img_records)

# Load text CSV
text_df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
text_df['file_id'] = text_df['File Name'].str.replace('.txt', '', regex=False).astype(int)
text_df['label']   = text_df['LABEL'].str.lower().str.strip()

# Join on file_id
paired = text_df.merge(img_df, on='file_id')[['file_id', 'Caption', 'label', 'img_path']]
paired = paired.dropna(subset=['Caption', 'img_path']).reset_index(drop=True)

print(f'Paired tweets: {len(paired)}')
print(paired['label'].value_counts())

Paired tweets: 4869
label
neutral     1771
positive    1646
negative    1452
Name: count, dtype: int64


## Joint train / val / test split

One split on the paired DataFrame — same parameters as all unimodal notebooks.
Both models train on the same tweets; the joint test set gives a direct apples-to-apples comparison.

In [3]:
trainval_df, test_df = train_test_split(
    paired, test_size=0.2, random_state=42, stratify=paired['label']
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.2, random_state=42, stratify=trainval_df['label']
)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

le = LabelEncoder()
le.fit(paired['label'])
print('Classes:', le.classes_)

y_train    = le.transform(train_df['label'])
y_val      = le.transform(val_df['label'])
y_test     = le.transform(test_df['label'])
y_trainval = le.transform(trainval_df['label'])

Train: 3116  Val: 779  Test: 974
Classes: ['negative' 'neutral' 'positive']


## Text preprocessing

Identical pipeline to `03_glove_rnn_text.ipynb`: lowercase, strip URLs/punctuation,
remove stopwords, tokenize + pad to MAX_LEN=50. Tokenizer fit on joint training set only.

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

train_text    = train_df['Caption'].apply(clean_text)
val_text      = val_df['Caption'].apply(clean_text)
test_text     = test_df['Caption'].apply(clean_text)
trainval_text = trainval_df['Caption'].apply(clean_text)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_text)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_txt    = encode(train_text)
X_val_txt      = encode(val_text)
X_test_txt     = encode(test_text)
X_trainval_txt = encode(trainval_text)

print('Text encoded. Shape:', X_train_txt.shape)

Text encoded. Shape: (3116, 50)


## Load GloVe and build embedding matrix

In [5]:
glove = {}
with open('../data/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype='float32')
print(f'GloVe vocab: {len(glove):,}')

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx < VOCAB_SIZE and word in glove:
        embedding_matrix[idx] = glove[word]
        hits += 1
print(f'GloVe coverage: {hits} / {min(VOCAB_SIZE, len(tokenizer.word_index))}')

GloVe vocab: 400,000
GloVe coverage: 6660 / 10000


## Load all images into memory

Images preprocessed with MobileNetV2's `preprocess_input` (scales to [-1, 1]).
Loaded once in paired-DataFrame order so indices align with text arrays.

In [6]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32')
    return mobilenet_preprocess(arr)

print(f'Loading {len(paired)} images...')
X_all_img = np.array([load_image(p) for p in paired['img_path']])
print(f'Loaded. Shape: {X_all_img.shape}  Memory: {X_all_img.nbytes / 1e9:.2f} GB')

# Slice by split index (paired_df index → positional index in X_all_img)
train_idx    = train_df.index
val_idx      = val_df.index
test_idx     = test_df.index
trainval_idx = trainval_df.index

X_train_img    = X_all_img[train_idx]
X_val_img      = X_all_img[val_idx]
X_test_img     = X_all_img[test_idx]
X_trainval_img = X_all_img[trainval_idx]

print(f'Train img: {X_train_img.shape}  Val: {X_val_img.shape}  Test: {X_test_img.shape}')

Loading 4869 images...
Loaded. Shape: (4869, 128, 128, 3)  Memory: 0.96 GB
Train img: (3116, 128, 128, 3)  Val: (779, 128, 128, 3)  Test: (974, 128, 128, 3)


## Train GloVe + LSTM text model on joint trainval

Architecture identical to `03_glove_rnn_text.ipynb`. Uses known best `LSTM_UNITS=32`
(determined on the unimodal text split) — no re-tuning to keep total runtime manageable.

In [7]:
t0 = time.time()

text_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], trainable=False),
    LSTM(LSTM_UNITS),
    Dense(3, activation='softmax')
])
text_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

text_model.fit(
    X_trainval_txt, y_trainval,
    epochs=150,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)],
    verbose=1
)

text_probs_test = text_model.predict(X_test_txt)
text_acc = accuracy_score(y_test, np.argmax(text_probs_test, axis=1))
print(f'\nText model accuracy on joint test set: {text_acc:.3f}')

Epoch 1/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4303 - loss: 1.0424
Epoch 2/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6185 - loss: 0.8626
Epoch 3/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6716 - loss: 0.7668
Epoch 4/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7073 - loss: 0.6968
Epoch 5/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7402 - loss: 0.6486
Epoch 6/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7587 - loss: 0.6039
Epoch 7/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7725 - loss: 0.5721
Epoch 8/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7854 - loss: 0.5524
Epoch 9/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7969 - loss: 0.5343
Epoch 10/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7985 - loss: 0.5272
Epoch 11/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8062 - loss: 0.5054
Epoch 12/150
122/122 ━━━━━━━━━━━━━━━━━

## Train MobileNetV2 image model on joint trainval

Architecture identical to `03_cnn_mobilenet.ipynb`. Uses known best `DENSE_UNITS=256`.
Uses `validation_split=0.2` on X_trainval for EarlyStopping (avoids reusing the fixed val set).

In [8]:
base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base.trainable = False

img_model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(DENSE_UNITS, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])
img_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

img_model.fit(
    X_trainval_img, y_trainval,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

img_probs_test = img_model.predict(X_test_img)
img_acc = accuracy_score(y_test, np.argmax(img_probs_test, axis=1))
print(f'\nImage model accuracy on joint test set: {img_acc:.3f}')

Epoch 1/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 19s 148ms/step - accuracy: 0.3383 - loss: 1.5287 - val_accuracy: 0.3877 - val_loss: 1.2108
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 136ms/step - accuracy: 0.3790 - loss: 1.2664 - val_accuracy: 0.3736 - val_loss: 1.1721
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 14s 144ms/step - accuracy: 0.4320 - loss: 1.1408 - val_accuracy: 0.3800 - val_loss: 1.1565
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 158ms/step - accuracy: 0.4637 - loss: 1.0902 - val_accuracy: 0.3851 - val_loss: 1.1495
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 16s 158ms/step - accuracy: 0.4843 - loss: 1.0525 - val_accuracy: 0.3928 - val_loss: 1.1494
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 157ms/step - accuracy: 0.5372 - loss: 1.0106 - val_accuracy: 0.4134 - val_loss: 1.1415
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 150ms/step - accuracy: 0.5719 - loss: 0.9616 - val_accuracy: 0.4069 - val_loss: 1.1445
Epoch 8/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 155ms/step - accuracy: 0.5985 - loss: 0.9231 - 

## Late fusion — average softmax probabilities

Both models output P(class | tweet) as a 3-dim softmax vector.
Equal-weight average: `fusion_probs = 0.5 * text_probs + 0.5 * img_probs`.
Final label is argmax of the fused probability vector.

In [9]:
fusion_probs = 0.5 * text_probs_test + 0.5 * img_probs_test
y_fusion     = np.argmax(fusion_probs, axis=1)
runtime      = time.time() - t0

fusion_acc  = accuracy_score(y_test, y_fusion)
report      = classification_report(y_test, y_fusion, target_names=le.classes_, output_dict=True)

print('=== Joint test set comparison ===')
print(f'  Text only  (GloVe+LSTM):  {text_acc:.3f}')
print(f'  Image only (MobileNetV2): {img_acc:.3f}')
print(f'  Fusion (avg):             {fusion_acc:.3f}')
print()
print(classification_report(y_test, y_fusion, target_names=le.classes_))

=== Joint test set comparison ===
  Text only  (GloVe+LSTM):  0.678
  Image only (MobileNetV2): 0.389
  Fusion (avg):             0.677

              precision    recall  f1-score   support

    negative       0.66      0.61      0.63       291
     neutral       0.63      0.65      0.64       354
    positive       0.74      0.77      0.75       329

    accuracy                           0.68       974
   macro avg       0.68      0.67      0.68       974
weighted avg       0.68      0.68      0.68       974



## Save models and metadata

In [10]:
text_model.save('../models/both/fits/fusion_text_glove_lstm.keras')
img_model.save('../models/both/fits/fusion_img_mobilenet.keras')

meta = {
    'model': 'multimodal_late_fusion',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime,
    'text_acc_joint_test': text_acc,
    'img_acc_joint_test': img_acc,
    'fusion_strategy': 'equal_average',
    'text_model': 'GloVe+LSTM (units=32)',
    'image_model': 'MobileNetV2 (dense=256)'
}

with open('../models/both/json/late_fusion_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved. Total runtime: {runtime:.0f}s ({runtime/60:.1f} min)')

Saved. Total runtime: 432s (7.2 min)
